In [19]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import scipy
import sklearn
cwd = Path('.')

In [20]:
data_path = cwd / 'data' / '2025_LoL_esports_match_data_from_OraclesElixir_imputated.csv'
data = pd.read_csv(data_path,index_col=0)
data_len = int(len(data) * 10 / 12)

C:\Users\victo\AppData\Local\Temp\ipykernel_12224\4154608675.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(data_path,index_col=0)


In [21]:
data = data.drop(data[data["position"] == "team"].index) 

In [23]:
pick_set = []
ban_set = []
for i in tqdm(range(0,len(data),5)):
    data_game = data.iloc[i:i+5]
    pick_set_game = []
    ban_set_game = []
    champs_in_game = data_game["champion"]
    if (data_game["ban1"].hasnans == False): #Some bans are missing for some games, these are assumed to be as a result of a team using all their time and missing the opputunity to ban and as such the game can sstill be kept
        ban_set_game.append(data_game.iloc[0]["ban1"]+"_banned")
    if (data_game["ban2"].hasnans == False):
        ban_set_game.append(data_game.iloc[0]["ban2"]+"_banned")
    if (data_game["ban3"].hasnans == False):
        ban_set_game.append(data_game.iloc[0]["ban3"]+"_banned")
    if (data_game["ban4"].hasnans == False):
        ban_set_game.append(data_game.iloc[0]["ban4"]+"_banned")
    if (data_game["ban5"].hasnans == False):
        ban_set_game.append(data_game.iloc[0]["ban5"]+"_banned")
    for champ in champs_in_game:
        champ_name = champ + "_picked"
        pick_set_game.append(champ_name)
    pick_set.append(pick_set_game)
    ban_set.append(np.unique(ban_set_game))






  0%|          | 0/20106 [00:00<?, ?it/s]

In [24]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_ary = te.fit(pick_set).transform(pick_set)
df_ap = pd.DataFrame(te_ary, columns=te.columns_)

In [25]:
df_ap

,Aatrox_picked,Ahri_picked,Akali_picked,Akshan_picked,Alistar_picked,Ambessa_picked,Amumu_picked,Anivia_picked,Annie_picked,Aphelios_picked,...,Yunara_picked,Yuumi_picked,Zaahen_picked,Zac_picked,Zed_picked,Zeri_picked,Ziggs_picked,Zilean_picked,Zoe_picked,Zyra_picked
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,True,False,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20101,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20102,False,False,False,False,False,True,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
20103,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
20104,True,True,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False


In [26]:
from mlxtend.frequent_patterns import apriori

frequent_itemsets = apriori(df_ap, min_support=0.01,use_colnames=True)

We need to use a very low support to get sets of a size greater than 1 since there are only ever 5 champions and very many combinations of them.

In [27]:
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

In [28]:
freq_greater_1 = frequent_itemsets[(frequent_itemsets['length'])>1]

In [29]:
freq_greater_1.sort_values(by=["support"])

,support,itemsets,length
88,0.010047,"(Ambessa_picked, Ahri_picked)",2
170,0.010047,"(K'Sante_picked, Nautilus_picked)",2
196,0.010096,"(Rakan_picked, Orianna_picked)",2
188,0.010146,"(Maokai_picked, Rell_picked)",2
195,0.010146,"(Nocturne_picked, Orianna_picked)",2
...,...,...,...
208,0.020790,"(Rumble_picked, Rell_picked)",2
153,0.022481,"(Leona_picked, Ezreal_picked)",2
212,0.022879,"(Varus_picked, Rell_picked)",2
145,0.024072,"(Rell_picked, Corki_picked)",2


Xayah and Rakan are a typical combo so seeing them with the highest support is good

In [30]:
from mlxtend.frequent_patterns import fpgrowth

In [31]:
frequent_itemsets_fpg = fpgrowth(df_ap, min_support=0.01,use_colnames=True)

In [32]:
frequent_itemsets_fpg['length'] = frequent_itemsets_fpg['itemsets'].apply(lambda x: len(x))
freq_greater_1_fpg = frequent_itemsets_fpg[(frequent_itemsets_fpg['length'])>1]
freq_greater_1_fpg.sort_values(by=["support"])

,support,itemsets,length
123,0.010047,"(K'Sante_picked, Nautilus_picked)",2
184,0.010047,"(Ambessa_picked, Ahri_picked)",2
100,0.010096,"(Rakan_picked, Orianna_picked)",2
93,0.010146,"(Maokai_picked, Rell_picked)",2
166,0.010146,"(Varus_picked, Viktor_picked)",2
...,...,...,...
192,0.020790,"(Rumble_picked, Rell_picked)",2
89,0.022481,"(Leona_picked, Ezreal_picked)",2
96,0.022879,"(Varus_picked, Rell_picked)",2
126,0.024072,"(Rell_picked, Corki_picked)",2


In [33]:
pick_ban_set = [] # First is champ selected, second is ban
for i in range(len(pick_set)):
    for champ in pick_set[i]:
        for ban in ban_set[i]:
            pick_ban_set.append([champ,ban])

In [34]:
te = TransactionEncoder()
te_ary = te.fit(pick_ban_set).transform(pick_ban_set)
df_ap_pb = pd.DataFrame(te_ary, columns=te.columns_)

In [88]:
frequent_itemsets_pb = fpgrowth(df_ap_pb, min_support=0.0005,use_colnames=True)
frequent_itemsets_pb['length'] = frequent_itemsets_pb['itemsets'].apply(lambda x: len(x))
freq_greater_1_pb = frequent_itemsets_pb[(frequent_itemsets_pb['length'])>1]
freq_greater_1_pb.sort_values(by=["support"])

,support,itemsets,length
309,0.000503,"(Sejuani_picked, Skarner_banned)",2
362,0.000503,"(Varus_banned, Viktor_picked)",2
253,0.000503,"(Jayce_banned, Leona_picked)",2
467,0.000505,"(Neeko_banned, Nautilus_picked)",2
354,0.000505,"(Ambessa_banned, Alistar_picked)",2
...,...,...,...
492,0.001121,"(Sion_picked, Gwen_banned)",2
303,0.001131,"(Vi_banned, Rell_picked)",2
443,0.001139,"(Varus_banned, Xin Zhao_picked)",2
439,0.001153,"(Vi_banned, Xin Zhao_picked)",2


In [36]:
frequent_itemsets_pb.iloc[0]["itemsets"]==set(["Corki_banned"])

True

In [25]:
print(frequent_itemsets_pb[frequent_itemsets_pb["itemsets"]==set(["Varus_banned"])]["support"])
frequent_itemsets_pb[frequent_itemsets_pb["itemsets"]==set(["Rell_picked"])]["support"]

86    0.03797
Name: support, dtype: float64


25    0.030006
Name: support, dtype: float64

In [89]:
common_pick_ban = freq_greater_1_pb.sort_values(by=["support"])

In [90]:
common_pick_ban

,support,itemsets,length
309,0.000503,"(Sejuani_picked, Skarner_banned)",2
362,0.000503,"(Varus_banned, Viktor_picked)",2
253,0.000503,"(Jayce_banned, Leona_picked)",2
467,0.000505,"(Neeko_banned, Nautilus_picked)",2
354,0.000505,"(Ambessa_banned, Alistar_picked)",2
...,...,...,...
492,0.001121,"(Sion_picked, Gwen_banned)",2
303,0.001131,"(Vi_banned, Rell_picked)",2
443,0.001139,"(Varus_banned, Xin Zhao_picked)",2
439,0.001153,"(Vi_banned, Xin Zhao_picked)",2


In [51]:
common_pick_ban.iloc[pick_ban_index]

support                           0.001005
itemsets    (Alistar_picked, Varus_banned)
length                                   2
Name: 206, dtype: object

In [69]:
for i in common_pick_ban.iloc[pick_ban_index]["itemsets"]:
    print(i)

Varus_banned
Rell_picked


In [71]:
common_pick_ban.iloc[pick_ban_index]["itemsets"]

frozenset({'Rell_picked', np.str_('Varus_banned')})

In [91]:
norms = []
for pick_ban_index in range(len(common_pick_ban)):
    k = 1
    for i in common_pick_ban.iloc[pick_ban_index]["itemsets"]:
        k = k * frequent_itemsets_pb[frequent_itemsets_pb["itemsets"]==set([i])]["support"].iloc[0]
    norm_support = common_pick_ban.iloc[pick_ban_index]["support"] / k
    norms.append(norm_support)
common_pick_ban["norm_support"] = norms

In [96]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
    print(common_pick_ban.sort_values(by=["norm_support"])[20:])

      support                               itemsets  length  norm_support
333  0.000589       (Alistar_picked, Kalista_banned)       2      0.919517
362  0.000503          (Varus_banned, Viktor_picked)       2      0.924869
268  0.000551         (Varus_banned, Orianna_picked)       2      0.925509
477  0.000505       (Kai'Sa_picked, Pantheon_banned)       2      0.926746
337  0.000691          (Azir_banned, Alistar_picked)       2      0.927040
487  0.000579               (Vi_banned, Sion_picked)       2      0.927299
287  0.000621          (K'Sante_picked, Gwen_banned)       2      0.929370
304  0.000873           (Rumble_banned, Rell_picked)       2      0.930883
257  0.000521           (Aurora_banned, Rell_picked)       2      0.940855
282  0.000545            (Braum_picked, Gwen_banned)       2      0.942419
255  0.000677            (Leona_picked, Gwen_banned)       2      0.945977
425  0.000577            (Jhin_picked, Varus_banned)       2      0.949635
454  0.000537           (

In [70]:
frequent_itemsets_pb[frequent_itemsets_pb["itemsets"]==set([i])]["support"]

25    0.030006
Name: support, dtype: float64

In [61]:
norms

[38   NaN
 86   NaN
 Name: support, dtype: float64,
 55   NaN
 67   NaN
 Name: support, dtype: float64,
 30   NaN
 86   NaN
 Name: support, dtype: float64,
 43   NaN
 86   NaN
 Name: support, dtype: float64,
 83   NaN
 89   NaN
 Name: support, dtype: float64,
 25   NaN
 89   NaN
 Name: support, dtype: float64,
 25   NaN
 55   NaN
 Name: support, dtype: float64,
 89    NaN
 144   NaN
 Name: support, dtype: float64,
 5    NaN
 25   NaN
 Name: support, dtype: float64,
 83   NaN
 86   NaN
 Name: support, dtype: float64,
 5    NaN
 83   NaN
 Name: support, dtype: float64,
 25   NaN
 86   NaN
 Name: support, dtype: float64]

In [26]:
0.03797 * 0.030006

0.0011393278199999999

In [30]:
0.001175/0.0011393278199999999

1.031309847239577

In [27]:
print(frequent_itemsets_pb[frequent_itemsets_pb["itemsets"]==set(["Poppy_banned"])]["support"])
frequent_itemsets_pb[frequent_itemsets_pb["itemsets"]==set(["Ambessa_picked"])]["support"]

55    0.029079
Name: support, dtype: float64


67    0.024005
Name: support, dtype: float64

In [29]:
0.001021/(0.029079*0.024005)

1.4626639728149646

Varus is not really a good counter for rumble according to https://u.gg/lol/champions/rumble/counter, though we are seeing Varus appear high so he may just be a high priority ban

In [113]:
te = TransactionEncoder()
te_ary = te.fit(ban_set).transform(ban_set)
df_ap_ban = pd.DataFrame(te_ary, columns=te.columns_)

In [116]:
frequent_itemsets_fpg = fpgrowth(df_ap_ban, min_support=0.1,use_colnames=True)
frequent_itemsets_fpg['length'] = frequent_itemsets_fpg['itemsets'].apply(lambda x: len(x))
#freq_greater_1_fpg = frequent_itemsets_fpg[(frequent_itemsets_fpg['length'])>1]
frequent_itemsets_fpg.sort_values(by=["support"])

,support,itemsets,length
1,0.103800,(Skarner),1
12,0.103850,(Neeko),1
2,0.104049,(Yone),1
4,0.109271,(Ambessa),1
10,0.115289,(Taliyah),1
7,0.117030,(Jayce),1
3,0.133045,(Kalista),1
6,0.144783,(Poppy),1
13,0.145031,(Pantheon),1
11,0.154780,(Azir),1


As expected based on above, Varus is just a high priority ban

Can look more into counter picks (IE make each entry the laners for a given game on opposing sides), seperate by whether the team won or not, add a league or patch to see if a champ is really popular in a certain patch or league (could also seperate those into their own subdatasets). Look at the class roles (Tank, Mage etc), llok more clearly at pick ban, IE make the variables "Varus_picked" and "Varus_banned". Normalize by popularity see above